# 04 - Combined attacks, held-out detectors and a quality check (v7)

**Three questions, all answered in one run on top of your last run (same Drive folder `quick2`):**
1. Does a **combined attack** (paraphrase first, then light word edits) beat every detector at once?
2. Do the results survive **held-out detectors** (a different model family than the one that wrote the text)?
3. Are the "evaded" texts still **good quality**? An LLM judge scores fluency and meaning (1 to 5).

The end result is one number per attack: **the share of texts that beat ALL detectors AND keep good quality**.

**Everything is already set up. You do not paste or edit anything.** Before you start: **Runtime -> Change runtime type -> T4 GPU**, and switch on notebook access for `HF_TOKEN` (key icon, left sidebar). Then run the cells from top to bottom, one at a time. Every stage saves its work, so you can stop any stage with the square stop button and run it again later: it continues where it stopped.

## 1. Log in

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
print("HF user:", whoami()["name"])

## 2. Get the code (always a fresh copy)
If Colab asks to restart the session, do it, then rerun cells 1 and 2.

In [ ]:
!rm -rf /content/sira-lab
!git clone -q https://github.com/deymonp/sira-lab.git /content/sira-lab
%cd /content/sira-lab
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

## 3. Settings
Colab will ask you to allow access to Google Drive. Allow it. Check that the last line says `True`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN = "/content/drive/MyDrive/sira_runs/quick2"     # same folder as before: texts, attacks and scores are re-used

import os
from sira_lab.config import load_config
from sira_lab.lab.run import run_generate, run_attack, run_evaluate
from sira_lab.lab.judge import run_judge
cfg = load_config("configs/lab_v7.yaml")
print("attacks:", cfg.lab.attacks)
print("detectors:", cfg.lab.detectors)
print("texts from the last run found:", os.path.exists(f"{RUN}/generated.jsonl"))

## 4. Stage 2: the combined attacks
Only the four new combined attacks are computed (about 320 rewrites). The paraphrase results from the last run are re-used as their first step. The cell prints its running time at the end. Please keep that number.

In [ ]:
import time
t0 = time.time()
run_attack(cfg, RUN)
print(f"Stage 2 took {(time.time() - t0) / 60:.1f} minutes")

## 5. Stage 3: score everything with all detectors
Scores from the last run are re-used, so only the new combined attacks and the two new held-out detectors are scored. The held-out detectors download a different small model family (SmolLM2). Progress is saved after each detector.

In [ ]:
import time, pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 300)
t0 = time.time()
df = run_evaluate(cfg, RUN)
print(f"Stage 3 took {(time.time() - t0) / 60:.1f} minutes")
display(df[["detector", "algo", "attack", "detect_rate", "changed_mean", "sim_median"]].round(3))

## 6. Stage 4: the quality judge
The 7B model reads each text and gives a fluency score and a meaning-preserved score (1 to 5). It also scores the untouched human and machine texts, as reference points.

In [ ]:
import time
t0 = time.time()
run_judge(cfg, RUN)
print(f"Stage 4 took {(time.time() - t0) / 60:.1f} minutes")

## 7. The result: success rate per attack
**success** = the share of texts that beat every detector AND get fluency and meaning of at least 4 from the judge. Higher is better. `evade_wm` is the watermark detector, `evade_zs` the two original zero-shot detectors, `evade_zs_ho` the two held-out ones.

In [ ]:
from sira_lab.lab.analysis import success_table, plot_success
st = success_table(RUN, min_fluency=cfg.lab.min_fluency, min_meaning=cfg.lab.min_meaning)
st.to_csv(f"{RUN}/success.csv", index=False)
plot_success(st, f"{RUN}/success.png")
cols = ["algo", "attack", "n", "evade_wm", "evade_zs", "evade_zs_ho", "evade_all", "quality_ok", "success", "success_lo", "success_hi", "fluency_mean", "meaning_mean"]
print(st[cols].sort_values(["algo", "success"], ascending=[True, False]).round(3).to_string(index=False))
display(Image(f"{RUN}/success.png"))

## 8. What to send me
Open the Drive folder `My Drive / sira_runs / quick2` in your browser, download these files and attach them in the chat:

`success.csv`, `quality.csv`, `results.csv`, `scores.csv`, `success.png`

Also tell me the three times printed under Stage 2, 3 and 4. When you are done, click **Runtime -> Disconnect and delete runtime**.

In [ ]:
!ls -la {RUN}